In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import pyreadr

In [5]:
RUTA_CENSO_ZONAL = '../data/PARQUET/Cartografía_censo2024_R13_Zonal.parquet'
RUTA_CASEN = "../data/rds/casen_rm.rds"

# Entradas

In [6]:
censo_zonal = gpd.read_parquet(RUTA_CENSO_ZONAL)
casen_raw = pyreadr.read_r(RUTA_CASEN)

# Se transforma la casen a DF
casen_raw = casen_raw[list(casen_raw.keys())[0]]

# Pre-procesamiento

Se acota el área del censo únicamenteutilizando zonas censales del Gran Santiago

In [8]:
# Condición
es_gs = (censo_zonal["PROVINCIA"] == 'SANTIAGO') | (
    censo_zonal["COMUNA"].isin(['SAN BERNARDO', 'PUENTE ALTO'])
)
# Se filtra el DF en base a la condición
censo_zonal = censo_zonal[es_gs].reset_index(drop=True)

Se extraen las variables de la CASEN que se van a utilizar 

In [12]:
# vairables que no se tocan para microsimular.
# En caso de añadir una nueva el procesamiento posterior cambia
VARS_FIJAS = ["estrato", "esc", "e6a", "edad", "sexo",
              "v13", "v13_propia", "v13_arrendada", "v13_cedida"]

# Esta es la que se modifica
VARIABLE_A_MICROSIMULAR = "ypchtrabcor"

vars_a_cargar = VARS_FIJAS + [VARIABLE_A_MICROSIMULAR]

# Se acota la CASEN únicamente a las variables a utilizar
casen = casen_raw[vars_a_cargar].copy()

del casen_raw # Para liberar memoria

# Se obtiene la comuna de cada persoba
casen["comuna"] = (casen["estrato"]//100).astype("Int64")
casen = casen.drop(columns="estrato")

casen

,esc,e6a,edad,sexo,v13,v13_propia,v13_arrendada,v13_cedida,ypchtrabcor,comuna
0,12.0,11.0,54.0,2.0,1.0,2.0,NaN,NaN,520000.0,13111
1,12.0,11.0,28.0,2.0,1.0,2.0,NaN,NaN,520000.0,13111
2,17.0,13.0,27.0,2.0,1.0,2.0,NaN,NaN,520000.0,13111
3,10.0,11.0,19.0,2.0,1.0,2.0,NaN,NaN,520000.0,13111
4,12.0,11.0,56.0,1.0,1.0,2.0,NaN,NaN,520000.0,13111
...,...,...,...,...,...,...,...,...,...,...
38669,8.0,7.0,56.0,2.0,1.0,2.0,NaN,NaN,540787.0,13115
38670,12.0,11.0,33.0,1.0,3.0,NaN,NaN,8.0,209696.0,13115
38671,14.0,12.0,31.0,2.0,3.0,NaN,NaN,8.0,209696.0,13115
38672,NaN,7.0,10.0,1.0,3.0,NaN,NaN,8.0,209696.0,13115


In [13]:
ZONA_ID = "OBJECTID"

COLS_SEXO = {"hombre": "n_hombres", "mujer": "n_mujeres"}
COLS_EDAD = {
    "edad_0_5": "n_edad_0_5",
    "edad_6_13": "n_edad_6_13",
    "edad_14_17": "n_edad_14_17",
    "edad_18_24": "n_edad_18_24",
    "edad_25_44": "n_edad_25_44",
    "edad_45_59": "n_edad_45_59",
    "edad_60_mas": "n_edad_60_mas",
}

COLS_EDUC = ["educ_nunca", "educ_primaria", "educ_secundaria", "educ_terciaria"]

COLS_TENENCIA = {
    "ten_propia_pagada": "n_tenencia_propia_pagada",
    "ten_propia_pagandose": "n_tenencia_propia_pagandose",
    "ten_arrendada_contrato": "n_tenencia_arrendada_contrato",
    "ten_arrendada_sin_contrato": "n_tenencia_arrendada_sin_contrato",
    "ten_cedida_trabajo": "n_tenencia_cedida_trabajo",
    "ten_cedida_familiar": "n_tenencia_cedida_familiar",
    "ten_otro": "n_tenencia_otro",
}

constraints = censo_zonal[[ZONA_ID, "CUT", "COMUNA", "n_per", "n_hog"]].copy()

for nombre,col in {**COLS_SEXO, **COLS_EDAD}.items():
    constraints[nombre] = censo_zonal[col]

# Para educación tendremos 4 niveles
constraints["educ_nunca"] = censo_zonal["n_cine_nunca_curso_primera_infancia"]
constraints["educ_primaria"] = (censo_zonal["n_cine_primaria"] + 
                               censo_zonal["n_cine_especial_diferencial"])
constraints["educ_secundaria"] = censo_zonal["n_cine_secundaria"]
constraints["educ_terciaria"] = censo_zonal["n_cine_terciaria_maestria_doctorado"]
